In [17]:
from Reporting_Functions import *
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

In [18]:
# Load the Excel files

df_1_DATA = load_excel(r"D:\Projects\Lux_Project_Intern\Data\1_DATA.xlsx")
df_2_LIST_OF_COLUMNS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\2_LIST_OF_COLUMNS.xlsx")
df_3_LOB = load_excel(r"D:\Projects\Lux_Project_Intern\Data\3_LOB.xlsx")
df_4_TESTS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\4_TESTS.xlsx")
df_5_FINANCIALS = pd.read_excel(r"D:\Projects\Lux_Project_Intern\Data\5_FINANCIALS.xlsx", header=[0, 1])

## #CLEAN

In [19]:
# Clean and align the columns of df_1_DATA based on the column names in df_2_LIST_OF_COLUMNS

df_1_DATA = clean_and_align_columns(df_1_DATA, df_2_LIST_OF_COLUMNS['Column Names'].tolist())

In [20]:
# clean LA_LOB, LA_STATUS columns from df_1_DATA

df_1_DATA = clean_lob_status(
    df_1_DATA,
    lob_col="LA_LOB",
    status_col="LA_STATUS",
    lob_choices=df_3_LOB["Line of business"].dropna().tolist(),
    status_choices=df_3_LOB["Status"].dropna().tolist(),
    threshold=75
)

## #DATA TESTS 

In [21]:
# test 1- check_missing

test1 = check_missing(df=df_1_DATA, col_name='LA_LOB')
test2 = check_missing(df=df_1_DATA, col_name='LA_LOSS_DATE')
print(test1)
print(test2)

{'column': 'LA_LOB', 'status': 'PASS', 'message': "No nulls in 'LA_LOB'", 'failed_rows': []}
{'column': 'LA_LOSS_DATE', 'status': 'PASS', 'message': "No nulls in 'LA_LOSS_DATE'", 'failed_rows': []}


In [22]:
# test 2- check_missing

test3, df_1_DATA= AS_AT_DATE_CLEAN(df=df_1_DATA, as_at_col='LA_AS_AT_DATE', payment_col='LA_PAYMENT_DATE')

test3

{'column': 'LA_AS_AT_DATE',
 'status': 'PARTIALLY_FIXED',
 'message': "4 nulls found in 'LA_AS_AT_DATE' -> 4 rows substituted using quarter-end of 'LA_PAYMENT_DATE'",
 'rows_fixed': 4,
 'remaining_issues': [0, 23, 80]}

In [23]:
# test 3- PAYMENT_DATE_CLEAN

test4, df_1_DATA= PAYMENT_DATE_CLEAN(df=df_1_DATA, payment_date_col='LA_PAYMENT_DATE', status_col='LA_STATUS', condition_value='PAID')

test4


{'column': 'LA_PAYMENT_DATE',
 'status': 'FAIL',
 'message': "1 dates wiped (status != 'PAID'); 2 PAID rows still missing a payment date",
 'wiped_count': 1,
 'missing_paid_dates': 2}

In [24]:
# test 4- PAYMENT_VS_LOSS_DATE_VALIDATE

test5, df_1_DATA= PAYMENT_VS_LOSS_DATE_VALIDATE(df= df_1_DATA, payment_date_col='LA_PAYMENT_DATE', loss_date_col='LA_LOSS_DATE', allow_same_day=True)

test5

{'column': 'LA_PAYMENT_DATE vs LA_LOSS_DATE',
 'status': 'PASS',
 'message': '36 rows checked; 47 skipped (missing date(s)); 0 invalid (payment date before loss date)',
 'checked_count': 36,
 'skipped_count': 47,
 'invalid_count': 0,
 'invalid_rows': Empty DataFrame
 Columns: [LA_AS_AT_DATE, LA_POLICY_NUMBER, LA_CLAIM_NUMBER, LA_LOB, LA_LOSS_DATE, LA_PAYMENT_DATE, LA_STATUS, LA_GROSS_AMOUNT, LA_CEDED_AMOUNT, PAYMENT_VS_LOSS_VALIDATION]
 Index: []}

In [25]:
# test 5- VALUATION_VS_LOSS_DATE_VALIDATE (valuation data should be given)

# test6, df_1_DATA= VALUATION_VS_LOSS_DATE_VALIDATE()

# test6

In [ ]:
# agg tests

tests_agg = tests_agg(t1=test1, t2=test2, t3=test3, t4=test4, t5=test5)

tests_table = pd.DataFrame(data=tests_agg).fillna("no test here")

In [ ]:
# export data to xlsx

tests_table.to_excel(r"D:\Projects\Lux_Project_Intern\Report_Exported\Tests_Report.xlsx")

## #FINANCIALS TEST

In [28]:
df_5_FINANCIALS.columns = [
    "LOB",
    "SUM_OF_GROSS_OS",
    "SUM_OF_CEDED_OS",
    "SUM_OF_GROSS_PAID",
    "SUM_OF_CEDED_PAID"
]

df_5_FINANCIALS

,LOB,SUM_OF_GROSS_OS,SUM_OF_CEDED_OS,SUM_OF_GROSS_PAID,SUM_OF_CEDED_PAID
0,Engineering,0.00,0.00,5377587.55,4472023.54
1,Fire,329537.72,250576.33,360037.22,3621554.19
2,Motor Comp,1075780.00,591879.40,51857.47,0.00


In [29]:
df_1_DATA['Gross_Or_Ceded'] = np.where(df_1_DATA['LA_GROSS_AMOUNT'] != 0, 'GROSS', 'CEDED')

df_summary = df_1_DATA.groupby('LA_LOB', as_index=False).agg(
    SUM_OF_GROSS_OS=('LA_GROSS_AMOUNT', lambda x: x[df_1_DATA.loc[x.index, 'LA_STATUS'] == 'OS'].sum()),
    SUM_OF_CEDED_OS=('LA_CEDED_AMOUNT', lambda x: x[df_1_DATA.loc[x.index, 'LA_STATUS'] == 'OS'].sum()),
    SUM_OF_GROSS_PAID=('LA_GROSS_AMOUNT', lambda x: x[df_1_DATA.loc[x.index, 'LA_STATUS'] == 'PAID'].sum()),
    SUM_OF_CEDED_PAID=('LA_CEDED_AMOUNT', lambda x: x[df_1_DATA.loc[x.index, 'LA_STATUS'] == 'PAID'].sum())
).rename(columns={'LA_LOB': 'LOB'}).fillna(0.0)

df_summary

,LOB,SUM_OF_GROSS_OS,SUM_OF_CEDED_OS,SUM_OF_GROSS_PAID,SUM_OF_CEDED_PAID
0,ENGINEERING,0.00,0.00,5377587.55,4472023.54
1,FIRE,329537.72,250576.33,360037.22,3621554.19
2,MOTOR_COMPREHENSIVE,1075780.00,591879.40,51857.47,0.00


In [ ]:
# calc if there is any diff

df_summary['SUM_OF_GROSS_OS_diff'] = df_summary['SUM_OF_GROSS_OS'] - df_5_FINANCIALS['SUM_OF_GROSS_OS']
df_summary['SUM_OF_CEDED_OS_diff'] = df_summary['SUM_OF_CEDED_OS'] - df_5_FINANCIALS['SUM_OF_CEDED_OS']
df_summary['SUM_OF_GROSS_PAID_diff'] = df_summary['SUM_OF_GROSS_PAID'] - df_5_FINANCIALS['SUM_OF_GROSS_PAID']
df_summary['SUM_OF_CEDED_PAID_diff'] = df_summary['SUM_OF_CEDED_PAID'] - df_5_FINANCIALS['SUM_OF_CEDED_PAID']

In [ ]:
# export to xlsx

df_summary.to_excel(r"D:\Projects\Lux_Project_Intern\Report_Exported\Financials_Test_Report.xlsx")